# Lab 1: Build a Restaurant Booking Agent

Every Friday, someone in my friend group drops the message: *"plans tonight?"*

And then begins the chaos. Two hours of "where do you want to go", "not that place again", "is it far from the metro?", until half the group gives up and orders Swiggy from home.

This notebook builds an AI agent that handles the entire process — detecting where you are, finding real restaurants, checking if they're open, and confirming a booking. You query it in plain English. It does the rest.

---

### What you'll build

A **restaurant booking agent** with 5 tools, wired into a **ReAct loop** using LangGraph and Gemini.

### What you'll learn

- What tools are and how an agent decides which one to call
- How the ReAct loop works (Reason → Act → Observe → repeat)
- How to connect real web search to an agent using Tavily
- How to trace every step of the agent's reasoning as it runs

### APIs needed (both free, no billing setup)

| API | What it's for | Link |
|-----|--------------|------|
| Google AI Studio | Gemini model | [aistudio.google.com](https://aistudio.google.com) |
| Tavily Search | Real restaurant data from the web | [app.tavily.com](https://app.tavily.com) |

Add both keys to a `.env` file before running:

```
GOOGLE_AI_STUDIO_API_KEY=your_key_here
TAVILY_API_KEY=your_key_here
```

---
## Part 1 — Setup

Install dependencies and load your API keys from `.env`.

In [1]:
!pip install -q langchain langchain-google-genai langgraph tavily-python python-dotenv requests

In [2]:
import os
import random
import string
import warnings
import requests
from datetime import datetime, timedelta

warnings.filterwarnings("ignore")  # suppress LangGraph deprecation warnings

from dotenv import load_dotenv
from tavily import TavilyClient
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

load_dotenv()

tavily = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))

print("Environment loaded.")

Environment loaded.


---
## Part 2 — The Tools

Think about what you actually do when you want to book a restaurant. You figure out your area, search what's there, look at the details, check if it's open at your time, and book. Each of those steps is a **tool** — a Python function the agent can call to get information from the real world.

The agent itself (the LLM) is just a brain. It can reason and decide, but it can't look anything up on its own. Tools are how it reaches out and grabs real data.

Here's the mapping:

| What you do as a human | Tool the agent uses |
|------------------------|---------------------|
| Figure out your area | `get_current_location` |
| Search for restaurants | `search_restaurants` |
| Check details (rating, cuisine, address) | `get_restaurant_details` |
| Check if it's open at your time | `check_availability` |
| Make the reservation | `make_booking` |

One important thing: **the docstring on each tool is not for you — it's for the LLM.** The agent reads the docstring to understand when and how to call that tool. Write it like you're explaining it to the model, not to a developer.

### Tool 1 — Detect location

The user might not say where they are. Before the agent can search for anything, it needs to know the city. This tool calls [ipinfo.io](https://ipinfo.io), a free IP geolocation API that needs no key. One call, city and country back as a string.

In [3]:
@tool
def get_current_location() -> str:
    """Detect the user's current city and country based on their IP address.
    Call this when the user hasn't told you where they are.

    Returns:
        City, region, and country (e.g., 'Bangalore, Karnataka, IN').
    """
    try:
        resp = requests.get("https://ipinfo.io/json", timeout=5)
        data = resp.json()
        city = data.get("city", "Unknown")
        region = data.get("region", "")
        country = data.get("country", "")
        return f"{city}, {region}, {country}"
    except Exception as e:
        return f"Could not detect location: {e}"

### Tool 2 — Search restaurants

This is where the agent does the actual searching. Most restaurant APIs (Yelp, Google Places) require billing setup. Instead, we use **Tavily** — a search API built specifically for LLM agents that returns clean web content instead of raw HTML.

`search_depth="advanced"` is important here. Basic search returns directory links. Advanced search digs into review pages and gets you actual names, addresses, and ratings. The agent gets back plain text and reasons over it — no structured JSON needed.

In [4]:
@tool
def search_restaurants(location: str, cuisine: str, budget: str = "any") -> str:
    """Search for real restaurants using Tavily web search.

    Args:
        location: City or area (e.g., 'Bangalore', 'downtown Mumbai', 'Manhattan')
        cuisine: Type of food (e.g., 'Italian', 'Indian', 'Japanese', 'Chinese')
        budget: 'low', 'medium', 'high', or 'any' (default)

    Returns:
        A list of real restaurants with names, addresses, ratings, and descriptions.
        Pick the most relevant ones and use their names with other tools.
    """
    try:
        budget_hint = f"{budget} budget" if budget != "any" else ""
        query = f"top {cuisine} restaurants in {location} {budget_hint} list with names addresses phone numbers ratings {datetime.now().year}".strip()

        result = tavily.search(
            query=query,
            search_depth="advanced",
            max_results=5,
            include_answer=True,
        )

        answer = result.get("answer", "")
        snippets = "\n\n".join(
            f"Source: {r['url']}\n{r['content']}"
            for r in result.get("results", [])
        )

        return (
            f"Search results for {cuisine} restaurants in {location}:\n\n"
            f"Summary: {answer}\n\n"
            f"Details:\n{snippets}"
        )
    except Exception as e:
        return f"Search failed: {e}"

### Tool 3 — Get restaurant details

Once the agent has a shortlist from the search, it can drill into a specific restaurant to get the address, phone number, and opening hours. Same Tavily approach — search the web, return the text, let the LLM parse it.

**When does it get called?** When the initial search results don't include enough detail about a specific place the agent wants to book. In Test 1,  Ottimo Cucina Italiana's details came through clearly in the search results so the agent skipped this and went straight to availability. Try querying a more obscure restaurant and you'll see it triggered.

In [5]:
@tool
def get_restaurant_details(restaurant_name: str, location: str) -> str:
    """Get detailed information about a specific restaurant — address, phone, hours, menu highlights.

    Args:
        restaurant_name: Full name of the restaurant (e.g., 'Trattoria Bella Italia')
        location: City or area where the restaurant is (e.g., 'Bangalore')

    Returns:
        Address, phone number, opening hours, cuisine type, and any other relevant details.
    """
    try:
        query = f"{restaurant_name} restaurant {location} address phone opening hours"

        result = tavily.search(
            query=query,
            search_depth="basic",
            max_results=3,
            include_answer=True,
        )

        answer = result.get("answer", "")
        snippets = "\n\n".join(
            f"Source: {r['url']}\n{r['content']}"
            for r in result.get("results", [])
        )

        return (
            f"Details for {restaurant_name} in {location}:\n\n"
            f"Summary: {answer}\n\n"
            f"From the web:\n{snippets}"
        )
    except Exception as e:
        return f"Could not fetch details: {e}"

### Tool 4 — Check availability

This is the step most people forget in the mental model: verifying the restaurant is actually open at the time you want. The tool fetches real opening hours from the web and hands them to the agent along with the day of the week — the agent then reasons about whether your requested time falls within those hours.

`resolve_date` is a small helper that turns "tonight" or "tomorrow" into an actual date, so the agent can include the day name (e.g. "Tuesday") in its search query.

**One honest caveat:** this tool searches for opening hours text and lets the LLM infer whether your time fits — it's not a real-time slot check against a booking system. Tavily might return outdated hours, and edge cases like lunch breaks or holiday hours can trip it up. In Lab 4, we replace this with Swiggy Dineout's `get_available_slots`, which returns actual availability. For Lab 1, this gets us 90% of the way there.

In [6]:
def resolve_date(date_str: str) -> datetime:
    """Resolve 'tonight', 'tomorrow', or 'YYYY-MM-DD' to a datetime."""
    s = date_str.strip().lower()
    if s in ("tonight", "today"):
        return datetime.now()
    if s == "tomorrow":
        return datetime.now() + timedelta(days=1)
    for fmt in ("%Y-%m-%d", "%d/%m/%Y", "%d %B %Y"):
        try:
            return datetime.strptime(date_str, fmt)
        except ValueError:
            continue
    return datetime.now()

In [7]:
@tool
def check_availability(restaurant_name: str, location: str, date: str, time: str) -> str:
    """Check if a restaurant is open at the requested date and time.
    Fetches real opening hours from the web and reasons about whether it's open.

    Args:
        restaurant_name: Full name of the restaurant
        location: City or area (e.g., 'Bangalore')
        date: Date of visit — 'tonight', 'tomorrow', or 'YYYY-MM-DD'
        time: Desired time in HH:MM format (e.g., '19:00', '20:30')

    Returns:
        Opening hours from the web plus the day and time being checked,
        so you can determine if the restaurant will be open.
    """
    try:
        dt = resolve_date(date)
        day_name = dt.strftime("%A")

        query = f"{restaurant_name} {location} opening hours {day_name}"

        result = tavily.search(
            query=query,
            search_depth="basic",
            max_results=3,
            include_answer=True,
        )

        answer = result.get("answer", "")
        snippets = "\n\n".join(
            f"Source: {r['url']}\n{r['content']}"
            for r in result.get("results", [])
        )

        return (
            f"Availability check for {restaurant_name} ({location}):\n"
            f"Requested: {day_name} at {time}\n\n"
            f"Opening hours summary: {answer}\n\n"
            f"From the web:\n{snippets}"
        )
    except Exception as e:
        return f"Could not check availability: {e}"

### Tool 5 — Make the booking

The agent should only call this after `check_availability` confirms the restaurant is open — notice that constraint is written directly in the docstring, so the model knows it. The booking is **simulated** for now: it generates a confirmation number and stores it in memory. Real booking APIs (Swiggy Dineout, OpenTable) require partnership access, which we wire in Lab 4 when we cover MCP.

> **Note on Lab 1 code style:** There's no try/except on every tool, no guard for empty Tavily results, and no explicit max iteration limit on the ReAct loop (LangGraph's default is 25 steps). This is intentional — Lab 1 keeps the code clean so the ReAct concepts stay front and centre. Error handling, retry logic, and explicit recursion limits show up in later labs.

In [8]:
BOOKINGS = {}

@tool
def make_booking(
    restaurant_name: str,
    location: str,
    date: str,
    time: str,
    party_size: int,
    guest_name: str,
) -> str:
    """Confirm a table booking. Only call this after check_availability confirms the restaurant is open.
    Note: booking is simulated — real integrations (OpenTable, Resy) require API partnerships.

    Args:
        restaurant_name: Full name of the restaurant
        location: City or area
        date: Date of booking (e.g., 'tonight', '2026-10-28')
        time: Time in HH:MM format (e.g., '19:00')
        party_size: Number of guests
        guest_name: Name for the reservation

    Returns:
        Booking confirmation with a unique confirmation number.
    """
    confirmation = "BR-" + "".join(random.choices(string.digits, k=4))

    BOOKINGS[confirmation] = {
        "restaurant": restaurant_name,
        "location": location,
        "date": date,
        "time": time,
        "party_size": party_size,
        "guest_name": guest_name,
    }

    return (
        f"✓ Booking confirmed! (call the restaurant to finalise your reservation)\n"
        f"  Restaurant: {restaurant_name}, {location}\n"
        f"  Date: {date} at {time}\n"
        f"  Party size: {party_size}\n"
        f"  Name: {guest_name}\n"
        f"  Confirmation #: {confirmation}"
    )

---
## Part 3 — The ReAct Loop

Now that we have 5 tools, we need something to orchestrate them. That's the agent. And the pattern it follows is called **ReAct**: Reason, Act, Observe, repeat.

Every iteration looks like this:

1. **Reason** — the LLM looks at the conversation so far and decides what it needs next. *"No location was given. I should detect it."*
2. **Act** — it picks a tool and calls it. `get_current_location()`.
3. **Observe** — it reads the result. *"Bengaluru, Karnataka, IN."*

Then it goes back to step 1. *"Got the location. Now search for Italian restaurants."* This continues until the agent has everything it needs to give a final answer — no more tools required.

`create_react_agent` from LangGraph handles this entire loop. You give it a model and a list of tools, and it manages the reasoning, tool dispatch, and message history automatically. Three lines of code, full ReAct loop.

In [9]:
model = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    google_api_key=os.getenv("GOOGLE_AI_STUDIO_API_KEY"),
    temperature=0,
)

tools = [
    get_current_location,
    search_restaurants,
    get_restaurant_details,
    check_availability,
    make_booking,
]

agent = create_react_agent(model, tools)

print("Agent ready.")

Agent ready.


---
## Part 4 — Run the Agent

The helper below streams the agent's execution so you can watch every ReAct step as it happens: the human message in, each AI thought, each tool call, each observation, and finally the answer.

This is the most important part of the tutorial. Don't just run it and read the output — watch the order. Notice when it decides to detect location vs. search vs. check availability. That sequence is the agent reasoning.

In [10]:
def run_agent(query: str):
    print("\n" + "=" * 65)
    print(f"USER: {query}")
    print("=" * 65)

    for chunk in agent.stream(
        {"messages": [("human", query)]},
        stream_mode="values",
    ):
        msg = chunk["messages"][-1]
        msg.pretty_print()
        print()

---
### Test 1 — No location given

The user doesn't say where they are. Watch the agent detect location first, then search, then check availability, then book. This is the full ReAct loop end to end.

In [11]:
run_agent(
    "Book me a good Italian restaurant tonight at 7:30pm for 2 people. My name is Kavya."
)


USER: Book me a good Italian restaurant tonight at 7:30pm for 2 people. My name is Kavya.
================================ Human Message =================================

Book me a good Italian restaurant tonight at 7:30pm for 2 people. My name is Kavya.

================================== Ai Message ==================================

[]
Tool Calls:
  get_current_location (call_161916)
 Call ID: call_161916
  Args:

================================= Tool Message =================================
Name: get_current_location

Bengaluru, Karnataka, IN

================================== Ai Message ==================================

[]
Tool Calls:
  search_restaurants (call_181138)
 Call ID: call_181138
  Args:
    location: Bengaluru
    cuisine: Italian

================================= Tool Message =================================
Name: search_restaurants

Search results for Italian restaurants in Bengaluru:

Summary: The sources mention several Italian restaurants in Bengaluru: IT

### Test 2 — Unusual time request

6 AM is before almost every restaurant opens. Watch how the agent handles this: it fetches real opening hours, reasons that 6 AM falls outside them, and either suggests an alternative time or explains what the actual hours are. The agent doesn't just fail — it backtracks.

In [12]:
run_agent(
    "Book me an Indian restaurant near me at 6:00am tomorrow for 3 people. Name is Kavya."
)


USER: Book me an Indian restaurant near me at 6:00am tomorrow for 3 people. Name is Kavya.
================================ Human Message =================================

Book me an Indian restaurant near me at 6:00am tomorrow for 3 people. Name is Kavya.

================================== Ai Message ==================================

[]
Tool Calls:
  get_current_location (call_162678)
 Call ID: call_162678
  Args:

================================= Tool Message =================================
Name: get_current_location

Bengaluru, Karnataka, IN

================================== Ai Message ==================================

[]
Tool Calls:
  search_restaurants (call_286135)
 Call ID: call_286135
  Args:
    location: Bengaluru
    cuisine: Indian

================================= Tool Message =================================
Name: search_restaurants

Search results for Indian restaurants in Bengaluru:

Summary: The sources mention several top Indian restaurants in Bengaluru.

### Test 3 — Ambiguous request

No cuisine, no time, no party size. Does the agent ask for the missing details or make assumptions and proceed? This tests how the model handles underspecified input — and shows you what information the agent considers required vs. optional.

In [13]:
run_agent(
    "Book me dinner tonight."
)


USER: Book me dinner tonight.
================================ Human Message =================================

Book me dinner tonight.

================================== Ai Message ==================================

[]
Tool Calls:
  get_current_location (call_209815)
 Call ID: call_209815
  Args:

================================= Tool Message =================================
Name: get_current_location

Bengaluru, Karnataka, IN

================================== Ai Message ==================================

[]
Tool Calls:
  search_restaurants (call_316824)
 Call ID: call_316824
  Args:
    location: Bengaluru

================================= Tool Message =================================
Name: search_restaurants

Error invoking tool 'search_restaurants' with kwargs {'location': 'Bengaluru'} with error:
 cuisine: Field required
 Please fix the error and try again.

================================== Ai Message ==================================

[]
Tool Calls:
  search_restaura

 Notice the self-correction: the first search call failed because cuisine is a required argument. The agent read the error and retried with cuisine: any — no instruction needed.

### Test 4 — Ask about a past booking

Now let's ask for something the agent has no way of knowing. This is the failure test — and it's the most important one.

In [14]:
run_agent(
    "Can you book the same restaurant we went to last month? Same time, same group size."
)


USER: Can you book the same restaurant we went to last month? Same time, same group size.
================================ Human Message =================================

Can you book the same restaurant we went to last month? Same time, same group size.

================================== Ai Message ==================================

[]
Tool Calls:
  get_current_location (call_204802)
 Call ID: call_204802
  Args:

================================= Tool Message =================================
Name: get_current_location

Bengaluru, Karnataka, IN

================================== Ai Message ==================================

[]
Tool Calls:
  search_restaurants (call_196619)
 Call ID: call_196619
  Args:
    cuisine: any
    location: Bengaluru

================================= Tool Message =================================
Name: search_restaurants

Search results for any restaurants in Bengaluru:

Summary: The sources mention several top restaurants in Bengaluru for 2026. Condé

---
## Part 5 — What you just watched

Every step above was the ReAct loop in action:

| What you saw | What it means |
|---|---|
| `AIMessage` with `tool_calls` | Agent is **reasoning** — decided which tool to call and with what arguments |
| `ToolMessage` | Tool ran and returned an **observation** — the agent reads this as text |
| `AIMessage` with no `tool_calls` | Agent has enough — this is the **final answer** |

Notice something important: the model never got clean structured data from a polished API. It got raw web text — snippets, paragraphs, mixed formatting — and it reasoned over that correctly every time. The LLM is the parser. That's what makes this approach work on real-world data without needing a purpose-built API for every source.

---

## Where this goes next

The agent found a real restaurant, checked real opening hours, and confirmed a booking. That's already more than a chatbot can do.

But run it again tomorrow with the same request and it starts from zero. It doesn't know you've been to Chianti before, that you prefer Italian, or that you never book places more than 20 minutes from Koramangala.

Imagine telling it "book somewhere for my usual Friday group" and it already knows what that means for you specifically — your preferences, your history, the things you always mention. That's what happens when you give an agent **memory**.

**In Lab 2, we plug in a memory layer using RAG. The agent will know your preferences, remember past bookings, and stop treating you like a stranger every time.**